In [43]:
import json
import os
from tqdm import tqdm
import re

test_path = '/data4/polyakov/instruction_tuning/ToolBench/data/answer/test_toollama_2_G3_inst_test_gpt_4_explanations'
ground_truth_test_path = '/data4/polyakov/instruction_tuning/ToolBench/data/test_data_merged/G3_inst.json'

In [44]:
len(ground_truth)

100

In [47]:
len(os.listdir(test_path))

100

In [46]:
ground_truth = json.load(open(ground_truth_test_path, 'r'))

responses = [0] * len(ground_truth)
for filename in os.listdir(test_path):
    query_number = int(filename.split('_')[0])
    data = json.load(open(test_path + '/' + filename, 'r'))
    responses[query_number] = data

for gt, resp in zip(ground_truth, responses):
    assert gt['query'] == resp['answer_generation']['query']

IndexError: list assignment index out of range

Подсчитаем процент случаев, в которых закончила с finish

In [38]:
win_count = 0
win_examples = []
lose_examples = []
for res in responses:
    if res['win']:
        win_count += 1
        win_examples.append(res)
    else:
        lose_examples.append(res)
win_count / len(responses)

0.86

In [39]:
lose_examples

[{'win': False,
  'try_count': 1,
  'trys': [{'chain': [{'is_terminal': False,
      'pruned': False,
      'finished': False,
      'depth': 1,
      'node_type': 'Thought',
      'description': '',
      'Elo': 1000.0,
      'child_count': 1,
      'expand_num': 0},
     {'is_terminal': False,
      'pruned': False,
      'finished': False,
      'depth': 2,
      'node_type': 'Action',
      'description': 'get_distance_by_city_state_country_for_great_circle_math_api',
      'Elo': 1000.0,
      'child_count': 1,
      'expand_num': 0},
     {'is_terminal': False,
      'pruned': False,
      'finished': False,
      'depth': 3,
      'node_type': 'Action Input',
      'description': '{\n  "country1": "US",\n  "country2": "US",\n  "state2": "CA",\n  "city2": "Buena Park",\n  "city1": "El Monte",\n  "state1": "AR"\n}',
      'Elo': 1000.0,
      'observation': '{"error": "", "response": "{\'cityName1\': \'Incorrect city/state/country\', \'stateName1\': \'Incorrect city/state/country\

Посмотрим на примеры, в которых win = True

In [36]:
win_examples[0]

{'win': True,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'metric_kilograms_for_body_mass_index_bmi_calculator',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "weight": 65,\n  "height": 160\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": "{\'bmi\': 0.0025390625, \'weight\': \'65\', \'height\': \'160\', \'weightCategory\': \'\'}"}',
     'observation_code': 0,
     'child_count': 1,
     'expand_num': 0,
     'io_state': {}},
    {'is_terminal': False,
     'pruned'

Посчитаем количество идентичных цепочек, как у Никиты

In [29]:
def list_api_called(item, keep_finish=False):
    api_list = []
    for message in item['answer_generation']['train_messages'][-1]:
        if message['role'] == 'assistant':
            curr_api = message['function_call']['name']
            if curr_api == 'Finish':
                if keep_finish:
                    api_list.append([curr_api, ''])
            else:
                api_list.append(curr_api.split('_for_')[::-1])
    return api_list

def list_api_called(item, keep_finish=False):
    trys = item['trys']
    assert len(trys) == 1
    chain = trys[0]['chain']
    api_list = []
    for message in chain:
        if message['node_type'] == 'Action':
            curr_api = message['description']
            if curr_api == 'Finish':
                if keep_finish:
                    api_list.append([curr_api, ''])
                else:
                    break
            else:
                api_list.append(curr_api.split('_for_')[::-1])
        if message['node_type'] == 'Action Input':
            api_list[-1].append(message['description'])
            api_list[-1].append(message['observation'])
    return api_list

In [30]:
count_identical_chains = 0
for idx, (gt, resp) in enumerate(zip(ground_truth, responses)):
    resp_api_list = list_api_called(resp)
    resp_api_list = [item[:2] for item in resp_api_list]
    if resp_api_list == gt['relevant APIs']:
        count_identical_chains += 1
        # print(idx)
        # print(resp_api_list)
        # print(gt['relevant APIs'])
        # print('#' * 100)
    else:
        print(idx)
        print(resp_api_list)
        print(gt['relevant APIs'])
        print('#' * 100)

4
[['body_mass_index_bmi_calculator', 'weight_category']]
[['body_mass_index_bmi_calculator', 'metric_kilograms'], ['body_mass_index_bmi_calculator', 'weight_category']]
####################################################################################################
6
[['body_mass_index_bmi_calculator', 'metric_kilograms']]
[['body_mass_index_bmi_calculator', 'metric_kilograms'], ['body_mass_index_bmi_calculator', 'weight_category']]
####################################################################################################
7
[['body_mass_index_bmi_calculator', 'weight_category']]
[['body_mass_index_bmi_calculator', 'metric_kilograms'], ['body_mass_index_bmi_calculator', 'weight_category']]
####################################################################################################
8
[['body_mass_index_bmi_calculator', 'weight_category']]
[['body_mass_index_bmi_calculator', 'metric_kilograms'], ['body_mass_index_bmi_calculator', 'weight_category']]
################

In [31]:
list_api_called(responses[64])

[['measurement_units_converter',
  'm_one_unit_of_measure_to_another',
  '{\n  "output_unit": "s",\n  "input_unit": "h",\n  "value": 101.027\n}',
  '{"error": "", "response": "{\'input\': {\'value\': \'101.027\', \'unit\': \'h\'}, \'output\': {\'value\': 363697.2, \'unit\': \'s\'}}"}']]

In [32]:
count_identical_chains / len(ground_truth)

0.36

Подсчитаем процент вызовов тулов c ошибками

In [33]:
error_messages = []
tool_calls_num = 0
error_messages_num = 0
for resp in responses:
    for step in list_api_called(resp):
        tool_calls_num += 1
        response = step[-1]
        print(response)
        # проверка на ошибку с в поле error
        error_message = re.findall('{"error": "([\s\S]*)", "response"', response)
        assert len(error_message) == 1
        error_message = error_message[0]
        if error_message:
            error_messages_num += 1
            error_messages.append((error_message, response))
            continue
        # проверка на ошибку с Incorrect
        if 'Incorrect' in response:
            error_messages_num += 1
            error_messages.append(("error hidden in tool output", response))
            continue
        # проверка на пустой респонс. если респонс пустой, то скорее всего распарсится json.loads
        try:
            json_response = json.loads(response)
            if not json_response['response']:
                error_messages_num += 1
                error_messages.append(('empty_response', response))
                continue
        except:
            pass

{"error": "", "response": "{'bmi': 0.0025390625, 'weight': '65', 'height': '160', 'weightCategory': ''}"}
{"error": "", "response": "{'bmi': '0.0025390625', 'weightCategory': 'Under Weight'}"}
{"error": "", "response": "{'bmi': 0.004571428571428572, 'weight': '140', 'height': '175', 'weightCategory': ''}"}
{"error": "", "response": "{'bmi': '0.004571428571428572', 'weightCategory': 'Under Weight'}"}
{"error": "", "response": "{'bmi': 0.002375, 'weight': '95', 'height': '200', 'weightCategory': ''}"}
{"error": "", "response": "{'bmi': '0.002375', 'weightCategory': 'Under Weight'}"}
{"error": "", "response": "{'bmi': 0.0039447731755424065, 'weight': '150', 'height': '195', 'weightCategory': ''}"}
{"error": "", "response": "{'bmi': '0.0039447731755424065', 'weightCategory': 'Under Weight'}"}
{"error": "", "response": "{'bmi': '115.0', 'weightCategory': 'Obese', 'ObeseClass': 'Class 3'}"}
{"error": "", "response": "{'bmi': 0.0031746031746031746, 'weight': '140', 'height': '210', 'weightCat

In [34]:
error_messages_num / tool_calls_num

0.27586206896551724

In [35]:
tool_calls_num

174

Опишем все ошибки

In [13]:
error_messages

[('Message error...',
  '{"error": "Message error...", "response": "{\'error\': {\'type\': \'DataException\', \'message\': \'no data\', \'code\': 800}}"}'),
 ('Message error...',
  '{"error": "Message error...", "response": "{\'error\': {\'type\': \'DataException\', \'message\': \'no data\', \'code\': 800}}"}'),
 ('Message error...',
  '{"error": "Message error...", "response": "{\'error\': {\'type\': \'DataException\', \'message\': \'no data\', \'code\': 800}}"}'),
 ('error hidden in tool output',
  '{"error": "", "response": "{\'cityName1\': \'Incorrect city/state/country\', \'stateName1\': \'Incorrect city/state/country\', \'country1\': \'Incorrect city/state/country\', \'latitude1\': \'NaN\', \'longitude1\': \'NaN\', \'cityName2\': \'Buena Park\', \'stateName2\': \'California\', \'country2\': \'US\', \'latitude2\': 33.870413, \'longitude2\': -117.9962165, \'calculatedDist\': {\'distance\': 0.0, \'uom\': \'mi\'}}"}'),
 ("Function executing from my_tools.Travel.great_circle_math_api.

Посмотрим на ошибки без учета частоты повторения тулов

In [212]:
errors_set = set()

for error in error_messages:
    errors_set.add(error[0])

In [215]:
errors_set

{"Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\\nget_distance_by_city_state_country() missing 1 required positional argument: 'country1'",
 "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\\nget_distance_by_city_state_country() missing 1 required positional argument: 'country2'",
 "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\\nget_distance_by_city_state_country() missing 2 required positional arguments: 'country1' and 'country2'",
 'Message error...',
 'empty_response',
 'error hidden in tool output'}

Посмотрим на все ошибки каждого типа

1) Message error

In [270]:
count = 0
for error in error_messages:
    if error[0] == 'Message error...':
        print(error[1])
        count += 1

{"error": "Message error...", "response": "{'error': 'Conversion from g to mi not supported.'}"}
{"error": "Message error...", "response": "{'error': 'Conversion from mi to mi not supported.'}"}
{"error": "Message error...", "response": "{'error': 'Conversion from mi to mi not supported.'}"}
{"error": "Message error...", "response": "{'error': 'Conversion from mi to mi not supported.'}"}
{"error": "Message error...", "response": "{'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}"}
{"error": "Message error...", "response": "{'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}"}
{"error": "Message error...", "response": "{'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}"}
{"error": "Message error...", "response": "{'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}"}
{"error": "Message error...", "response": "{'error': {'type': 'DataException', 'message': 'no data', 'code': 800}}"}
{"error": "Message error

In [271]:
count / tool_calls_num

0.09226190476190477

Message error есть двух типов

1) conversion not supported

Посмотрим на то, в каких примерах модель попыталась вызвать этот тул

In [228]:
flag = False
responses_message_error = []
for resp in responses:
    for step in list_api_called(resp):
        response = step[-1]
        if 'not supported' in response:
            responses_message_error.append(resp)
            break
            

In [235]:
responses_message_error[3]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_watches_by_brand_family_model_for_watch_database',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "brandname": "A. Lange & Söhne",\n  "model": "Cabaret 107.031",\n  "family": "A. Lange & Söhne"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": ""}',
     'observation_code': 0,
     'child_count': 1,
     'expand_num': 0,
     'io_state': {}},
    {'is_terminal': False,
     'pruned': False,
     'finished

2. DataException

In [236]:
responses_message_error_dataexcept = []
for resp in responses:
    for step in list_api_called(resp):
        response = step[-1]
        if 'DataException' in response:
            responses_message_error_dataexcept.append(resp)
            break

In [246]:
responses_message_error_dataexcept[6]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'genre_for_deezer',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "is_id": "161755"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "Message error...", "response": "{\'error\': {\'type\': \'DataException\', \'message\': \'no data\', \'code\': 800}}"}',
     'observation_code': 11,
     'child_count': 1,
     'expand_num': 0,
     'io_state': {}},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
    

3. Empty response

In [272]:
count = 0
for error in error_messages:
    if error[0] == 'empty_response':
        count += 1
        print(error[1])

{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}
{"error": "", "response": ""}


In [273]:
count / tool_calls_num

0.07738095238095238

In [250]:
responses_empty = []
for resp in responses:
    for step in list_api_called(resp):
        response = step[-1]
        if '{"error": "", "response": ""}' in response:
            responses_empty.append(resp)
            break

In [254]:
responses_empty[2]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_media_links_for_watch_database',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "watchid": "2102.50.00"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": ""}',
     'observation_code': 0,
     'child_count': 1,
     'expand_num': 0,
     'io_state': {}},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 4,
     'node_type': 'Thought',
     'description': '',
     'Elo'

4. Error hidden in tool output

In [274]:
count = 0
for error in error_messages:
    if error[0] == 'error hidden in tool output':
        print(error[1])
        count += 1

{"error": "", "response": "{'cityName1': 'Incorrect city/state/country', 'stateName1': 'Incorrect city/state/country', 'country1': 'Incorrect city/state/country', 'latitude1': 'NaN', 'longitude1': 'NaN', 'cityName2': 'Buena Park', 'stateName2': 'California', 'country2': 'US', 'latitude2': 33.870413, 'longitude2': -117.9962165, 'calculatedDist': {'distance': 0.0, 'uom': 'mi'}}"}
{"error": "", "response": "{'cityName1': 'Incorrect city/state/country', 'stateName1': 'Incorrect city/state/country', 'country1': 'Incorrect city/state/country', 'latitude1': 'NaN', 'longitude1': 'NaN', 'cityName2': 'Buena Park', 'stateName2': 'California', 'country2': 'US', 'latitude2': 33.870413, 'longitude2': -117.9962165, 'calculatedDist': {'distance': 0.0, 'uom': 'mi'}}"}
{"error": "", "response": "{'cityName1': 'Incorrect city/state/country', 'stateName1': 'Incorrect city/state/country', 'country1': 'Incorrect city/state/country', 'latitude1': 'NaN', 'longitude1': 'NaN', 'cityName2': 'Buena Park', 'stateN

In [275]:
count / tool_calls_num

0.017857142857142856

In [265]:
responses_hidden = []
for resp in responses:
    for step in list_api_called(resp):
        response = step[-1]
        if 'Incorrect' in response:
            responses_hidden.append(resp)
            break

In [269]:
responses_hidden[2]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_distance_by_city_state_country_for_great_circle_math_api',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "country1": "US",\n  "state1": "AZ",\n  "city1": "Berkeley",\n  "city2": "CA",\n  "state2": "CA",\n  "city2": "Alameda"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\\nget_distance_by_city_state_c

4. Missing or non-existent arguments

In [276]:
count = 0
for error in error_messages:
    if 'Function executing' in error[0] :
        print(error[1])
        count += 1

{"error": "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\nget_distance_by_city_state_country() missing 1 required positional argument: 'country1'", "response": ""}
{"error": "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\nget_distance_by_city_state_country() missing 2 required positional arguments: 'country1' and 'country2'", "response": ""}
{"error": "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\nget_distance_by_city_state_country() missing 2 required positional arguments: 'country1' and 'country2'", "response": ""}
{"error": "Function executing from my_tools.Travel.great_circle_math_api.api import get_distance_by_city_state_country error...\nget_distance_by_city_state_country() missing 2 required positional arguments: 'country1' and 'country2'", "response": ""}
{"error": "F

In [277]:
count / tool_calls_num

0.050595238095238096

In [278]:
9.2 + 7.7 + 1.8 + 5.1

23.799999999999997

Попробуем сохранить примеры с message error

In [281]:
flag = False
responses_message_error = []
for resp in responses:
    for step in list_api_called(resp):
        response = step[-1]
        if 'not supported' in response:
            responses_message_error.append(resp)
            break

In [282]:
responses_message_error[0]

{'win': False,
 'try_count': 1,
 'trys': [{'chain': [{'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 1,
     'node_type': 'Thought',
     'description': '',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 2,
     'node_type': 'Action',
     'description': 'get_distance_by_city_state_country_for_great_circle_math_api',
     'Elo': 1000.0,
     'child_count': 1,
     'expand_num': 0},
    {'is_terminal': False,
     'pruned': False,
     'finished': False,
     'depth': 3,
     'node_type': 'Action Input',
     'description': '{\n  "country1": "US",\n  "country2": "US",\n  "state2": "CA",\n  "city2": "San Bernardino",\n  "city1": "Norwalk",\n  "state1": "AK"\n}',
     'Elo': 1000.0,
     'observation': '{"error": "", "response": "{\'cityName1\': \'Incorrect city/state/country\', \'stateName1\': \'Incorrect city/state/country\', \'country1\': \'Incorr

In [285]:
json.dump(responses_message_error, open('responses_message_error.json', 'w'))